In [2]:
import pathlib
from pathlib import Path
import pandas as pd
import numpy as np
import time
import warnings
import os

pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

# --- PATH  CONFIGURATION ---
CURRENT_PATH = Path.cwd()

if CURRENT_PATH.name == 'notebooks':
    PROJECT_ROOT = CURRENT_PATH.parent
else:
    PROJECT_ROOT = CURRENT_PATH

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROC_DIR = PROJECT_ROOT / "data" / "processed"
PROC_DIR.mkdir(exist_ok=True, parents=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw Data Directory: {RAW_DIR}")

Project Root: e:\Tese\idletime
Raw Data Directory: e:\Tese\idletime\data\raw


In [3]:
# --- CELL 2: Identify Main CSV Files by Size ---

print("Scanning directory for largest CSV files...")

csv_files = []
for file_path in RAW_DIR.rglob("*.csv"):
    size_mb = file_path.stat().st_size / (1024 * 1024)
    csv_files.append({
        "path": file_path,
        "name": file_path.name,
        "size_mb": size_mb,
        "folder": file_path.parent.name
    })

df_files = pd.DataFrame(csv_files)

if not df_files.empty:
    df_files = df_files.sort_values(by="size_mb", ascending=False)
    
    print(f"\nTop 10 Largest Files Found:")
    print("-" * 100)
    print(f"{'File Name':<40} | {'Folder':<30} | {'Size (MB)':>15}")
    print("-" * 100)
    
    for _, row in df_files.head(10).iterrows():
        print(f"{row['name']:<40} | {row['folder']:<30} | {row['size_mb']:>15.2f} MB")
        
    target_files = df_files.head(5)['path'].tolist()
else:
    print("No CSV files found.")

Scanning directory for largest CSV files...

Top 10 Largest Files Found:
----------------------------------------------------------------------------------------------------
File Name                                | Folder                         |       Size (MB)
----------------------------------------------------------------------------------------------------
Fendt 722.csv                            | Fendt 722 - Kopie              |         3400.84 MB
Fendt 314.csv                            | Fendt 314 - Kopie              |         2859.89 MB
Fendt 724.csv                            | Fendt 724 - Kopie              |         1195.61 MB
Fendt 820.csv                            | Fendt 820 - Kopie              |          813.50 MB
Fendt 211.csv                            | Fendt 211 - Kopie              |          519.10 MB
Field_7.csv                              | Seed drill combination         |          298.84 MB
Field_3.csv                              | Seed drill combinati

In [4]:
# --- CELL 3: Inspect Headers and Format of All Target Files ---

if 'target_files' in locals() and target_files:
    for i, file_path in enumerate(target_files):
        print(f"\n{'='*100}")
        print(f"[{i+1}/{len(target_files)}] {file_path.name}  ({file_path.stat().st_size / 1024**2:.1f} MB)")
        print(f"{'='*100}")
        
        # Raw peek
        print("\n--- First 3 Lines (Raw Text) ---")
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            for _ in range(3):
                print(f.readline().strip())
        
        # Pandas read
        try:
            df_sample = pd.read_csv(file_path, nrows=5, sep=',')
            print(f"\nShape (5-row sample): {df_sample.shape}")
            print(f"Columns ({len(df_sample.columns)}): {list(df_sample.columns)}")
            print(f"Dtypes:\n{df_sample.dtypes}\n")
            display(df_sample.head(2))
        except Exception as e:
            print(f"Error reading file: {e}")
else:
    print("'target_files' not defined. Run Cell 2 first.")


[1/5] Fendt 722.csv  (3400.8 MB)

--- First 3 Lines (Raw Text) ---
Time_(s),index_[-],AccelPedalPos1_(%),ActualEngPercentTorque_(%),CourseOverGround_(°),EngFuelRate_(L/h),EngPercentLoadAtCurrentSpeed_(%),EngRequestedSpeed_SpeedLimit_(RPM),EngSpeed_(RPM),EstimatedCurvature_(1/km),FrontHitchInworkIndication_[-],FrontHitchPos_(%),GroundBasedMachineDistance_(m),GroundBasedMachineSpeed_(m/s),Latitude_(°),LeftStopLights_[-],Longitude_(°),MachineSelectedDistance_(m),MachineSelectedSpeed_(m/s),RearDraft_(N),RearHitchInWorkIndication_[-],RearHitchPosition_[-],RearNominalLowerLinkForce_(%),RearPTOOutputShaftSpeed_(RPM),WheelBasedMachineDistance_(m),WheelBasedMachineSpeed_(m/s),WheelBasedSpeed_(RPM),Cluster_[-],WorkType_[-],Tractor_Model_[-],Implement_Model_[-],Implement_Width_(m),Status_[-]
0 days 00:00:00,0,3.2,0.0,170.13981726409796,4.3,1.0,750.0,801.0,-4.75,0.0,250.0,398.878,0.083,48.1836093,0.0,11.3306903,398.894,0.083,20110.0,1.0,76.4,21.6,0.0,398.834,0.083,83.0,0.0,Seed drill combination,

,Time_(s),index_[-],AccelPedalPos1_(%),ActualEngPercentTorque_(%),CourseOverGround_(°),EngFuelRate_(L/h),EngPercentLoadAtCurrentSpeed_(%),EngRequestedSpeed_SpeedLimit_(RPM),EngSpeed_(RPM),EstimatedCurvature_(1/km),FrontHitchInworkIndication_[-],FrontHitchPos_(%),GroundBasedMachineDistance_(m),GroundBasedMachineSpeed_(m/s),Latitude_(°),LeftStopLights_[-],Longitude_(°),MachineSelectedDistance_(m),MachineSelectedSpeed_(m/s),RearDraft_(N),RearHitchInWorkIndication_[-],RearHitchPosition_[-],RearNominalLowerLinkForce_(%),RearPTOOutputShaftSpeed_(RPM),WheelBasedMachineDistance_(m),WheelBasedMachineSpeed_(m/s),WheelBasedSpeed_(RPM),Cluster_[-],WorkType_[-],Tractor_Model_[-],Implement_Model_[-],Implement_Width_(m),Status_[-]
0,0 days 00:00:00,0,3.2,0.0,170.139817,4.30,1.0,750.0,801.0,-4.75,0.0,250.0,398.878,0.083,48.183609,0.0,11.33069,398.894,0.083,20110.0,1.0,76.4,21.6,0.0,398.834,0.083,83.0,0.0,Seed drill combination,Fendt 722,Amazone AD 303 & Amazone KG 302,3.0,Driving Off-Road
1,0 days 00:00:00.100000,1,3.2,0.0,170.134088,4.35,1.0,750.0,799.0,-4.75,0.0,250.0,398.894,0.083,48.183609,0.0,11.33069,398.894,0.083,19410.0,1.0,73.6,20.8,0.0,398.834,0.083,83.0,NaN,not working,Fendt 722,Amazone AD 303 & Amazone KG 302,3.0,Driving Off-Road



[2/5] Fendt 314.csv  (2859.9 MB)

--- First 3 Lines (Raw Text) ---
Time_(s),index_[-],EngSpeed_(RPM),ActualEngPercentTorque_(%),DriversDemandEngPercentTorque_(%),EngFuelRate_(L/h),EngReferenceTorque_[Nm],Longitude_(°),Latitude_(°),EngPercentLoadAtCurrentSpeed_(%),AccelPedalPos1_(%),EngCoolantPress_(kPa),EngOilPress_(kPa),EngFuelDeliveryPress_(kPa),EngCoolantTemp_(°C),EngIntakeManifold1Temp_(°C),EngIntakeManifold1Press_(kPa),FrontAxleSpeed_(km/h),SpeedOverGround_(m/s),CourseOverGround_(°),AmbientAirTemp_(°C),Altitude_(m),DEFDoser1AbsPress_(kPa),DEFActualDosingQuantity_(g/h),EngRequestedSpeed_SpeedLimit_(RPM),GroundBasedImplementDistance_[mm],GroundBasedImplementSpeed_[mm/s],Volume_(%),WheelBasedSpeed_(RPM),WheelBasedVehicleSpeed _(km/h),EngChargeAirCooler1OutletTempEngFuelTemp1,NominalFrictionPercentTorque,RearPTOOutputShaftSpeed_(RPM),EstimatedCurvature_(1/km),RearHitchPosition_[-],RearHitchInWorkIndication_[-],RearNominalLowerLinkForce_(%),RearDraft_(N),GroundBasedMachineSpeed_(m/s),

,Time_(s),index_[-],EngSpeed_(RPM),ActualEngPercentTorque_(%),DriversDemandEngPercentTorque_(%),EngFuelRate_(L/h),EngReferenceTorque_[Nm],Longitude_(°),Latitude_(°),EngPercentLoadAtCurrentSpeed_(%),AccelPedalPos1_(%),EngCoolantPress_(kPa),EngOilPress_(kPa),EngFuelDeliveryPress_(kPa),EngCoolantTemp_(°C),EngIntakeManifold1Temp_(°C),EngIntakeManifold1Press_(kPa),FrontAxleSpeed_(km/h),SpeedOverGround_(m/s),CourseOverGround_(°),AmbientAirTemp_(°C),Altitude_(m),DEFDoser1AbsPress_(kPa),DEFActualDosingQuantity_(g/h),EngRequestedSpeed_SpeedLimit_(RPM),GroundBasedImplementDistance_[mm],GroundBasedImplementSpeed_[mm/s],Volume_(%),WheelBasedSpeed_(RPM),WheelBasedVehicleSpeed _(km/h),EngChargeAirCooler1OutletTempEngFuelTemp1,NominalFrictionPercentTorque,RearPTOOutputShaftSpeed_(RPM),EstimatedCurvature_(1/km),RearHitchPosition_[-],RearHitchInWorkIndication_[-],RearNominalLowerLinkForce_(%),RearDraft_(N),GroundBasedMachineSpeed_(m/s),GroundBasedMachineDistance_(m),MachineSelectedSpeed_(m/s),WheelBasedMachineSpeed_(m/s),FrontNominalLowerLinkForce_(%),FrontDraft_(N),LeftStopLights_[-],FrontHitchPos_(%),FrontHitchInworkIndication_[-],RightStopLight,Cluster_[-],WorkType_[-],Tractor_Model_[-],Implement_Model_[-],Implement_Width_(m),Status_[-]
0,0 days 00:00:00,0,268.0,54.0,0.0,1.250000,650.0,11.694324,48.405582,106.0,2.4,0.0,8.0,188.0,69.15625,215.0,510.0,0.0,0.0,261.63544,26.53125,521.28840,96.0,0.0,750.0,0.0,0,64.8,0.0,0.0,NaN,NaN,NaN,-6.75,2.0,0.0,-7.2,-4272.0180,0.0,0.0,0.0,0.0,104.0,335350.0,0.0,NaN,NaN,NaN,NaN,not working,Fendt 314,Rauch Axis,15,Driving Off-Road
1,0 days 00:00:00.100000,1,437.5,47.0,0.0,1.738414,650.0,11.694324,48.405582,109.0,2.4,0.0,8.0,188.0,69.15625,215.0,510.0,0.0,0.0,261.63260,26.53125,521.28845,96.0,0.0,750.0,0.0,0,64.8,0.0,0.0,NaN,NaN,NaN,-6.75,2.0,0.0,-7.2,-4288.2476,0.0,0.0,0.0,0.0,104.0,335350.0,0.0,NaN,NaN,NaN,NaN,not working,Fendt 314,Rauch Axis,15,Driving Off-Road



[3/5] Fendt 724.csv  (1195.6 MB)

--- First 3 Lines (Raw Text) ---
Time_(s),index_[-],EngSpeed_(RPM),ActualEngPercentTorque_(%),DriversDemandEngPercentTorque_(%),EngFuelRate_(L/h),EngReferenceTorque_[Nm],Longitude_(°),Latitude_(°),EngPercentLoadAtCurrentSpeed_(%),AccelPedalPos1_(%),EngCoolantPress_(kPa),EngOilPress_(kPa),EngFuelDeliveryPress_(kPa),EngCoolantTemp_(°C),EngIntakeManifold1Temp_(°C),EngIntakeManifold1Press_(kPa),FrontAxleSpeed_(km/h),SpeedOverGround_(m/s),CourseOverGround_(°),AmbientAirTemp_(°C),Altitude_(m),DEFDoser1AbsPress_(kPa),DEFActualDosingQuantity_(g/h),EngRequestedSpeed_SpeedLimit_(RPM),GroundBasedImplementDistance_[mm],GroundBasedImplementSpeed_[mm/s],NominalFrictionPercentTorque,Volume_(%),WheelBasedSpeed_(RPM),WheelBasedVehicleSpeed _(km/h),EngChargeAirCooler1OutletTempEngFuelTemp1,RearPTOOutputShaftSpeed_(RPM),EstimatedCurvature_(1/km),RearHitchPosition_[-],RearHitchInWorkIndication_[-],RearNominalLowerLinkForce_(%),RearDraft_(N),GroundBasedMachineSpeed_(m/s),

,Time_(s),index_[-],EngSpeed_(RPM),ActualEngPercentTorque_(%),DriversDemandEngPercentTorque_(%),EngFuelRate_(L/h),EngReferenceTorque_[Nm],Longitude_(°),Latitude_(°),EngPercentLoadAtCurrentSpeed_(%),AccelPedalPos1_(%),EngCoolantPress_(kPa),EngOilPress_(kPa),EngFuelDeliveryPress_(kPa),EngCoolantTemp_(°C),EngIntakeManifold1Temp_(°C),EngIntakeManifold1Press_(kPa),FrontAxleSpeed_(km/h),SpeedOverGround_(m/s),CourseOverGround_(°),AmbientAirTemp_(°C),Altitude_(m),DEFDoser1AbsPress_(kPa),DEFActualDosingQuantity_(g/h),EngRequestedSpeed_SpeedLimit_(RPM),GroundBasedImplementDistance_[mm],GroundBasedImplementSpeed_[mm/s],NominalFrictionPercentTorque,Volume_(%),WheelBasedSpeed_(RPM),WheelBasedVehicleSpeed _(km/h),EngChargeAirCooler1OutletTempEngFuelTemp1,RearPTOOutputShaftSpeed_(RPM),EstimatedCurvature_(1/km),RearHitchPosition_[-],RearHitchInWorkIndication_[-],RearNominalLowerLinkForce_(%),RearDraft_(N),GroundBasedMachineSpeed_(m/s),GroundBasedMachineDistance_(m),MachineSelectedSpeed_(m/s),WheelBasedMachineSpeed_(m/s),FrontNominalLowerLinkForce_(%),FrontDraft_(N),LeftStopLights_[-],FrontHitchPos_(%),FrontHitchInworkIndication_[-],RightStopLight,Cluster_[-],WorkType_[-],Tractor_Model_[-],Implement_Model_[-],Implement_Width_(m),Status_[-]
0,0 days 00:00:00,0,241.5,52.0,46.0,2.5,1189.0,11.694086,48.40569,99.0,0.0,510.0,0.0,616.0,12.0,13.0,94.0,0.0,0.0,169.71582,11.28125,521.6597,96.0,0.0,800.0,0.0,0,13.0,100.0,0.0,0.0,NaN,NaN,0.25,100.0,0.0,0.0,-620.0,0.0,0.0,1.0,0.0,104.0,335350.0,0.0,NaN,NaN,NaN,NaN,Transport,Fendt 724,Fliegl Agroliner 16 t,unknown,Driving On-Road
1,0 days 00:00:00.100000,1,241.5,52.0,46.0,2.5,1189.0,11.694086,48.40569,99.0,0.0,510.0,0.0,616.0,12.0,13.0,94.0,0.0,0.0,169.71582,11.28125,521.6597,96.0,0.0,800.0,0.0,0,13.0,100.0,0.0,0.0,NaN,NaN,0.25,100.0,0.0,0.0,-620.0,0.0,0.0,1.0,0.0,104.0,335350.0,0.0,NaN,NaN,NaN,NaN,Transport,Fendt 724,Fliegl Agroliner 16 t,unknown,Driving On-Road



[4/5] Fendt 820.csv  (813.5 MB)

--- First 3 Lines (Raw Text) ---
Time_(s),index_[-],AccelPedalPos1_(%),ActualEngPercentTorque_(%),Altitude_(m),EngFuelRate_(L/h),EngPercentLoadAtCurrentSpeed_(%),EngRequestedSpeed_SpeedLimit_(RPM),EngSpeed_(RPM),FrontHitchInworkIndication_[-],FrontHitchPos_(%),Latitude_(°),Longitude_(°),RearDraft_(N),RearHitchInWorkIndication_[-],RearHitchPosition_[-],WheelBasedMachineDistance_(m),WheelBasedMachineSpeed_(m/s),WheelBasedSpeed_(RPM),RearPTOOutputShaftSpeed_(RPM),Cluster_[-],WorkType_[-],Tractor_Model_[-],Implement_Model_[-],Implement_Width_(m),Status_[-]
0 days 00:00:00,0,2.799999952316284,26.0,575.1,4.900000095367432,49.0,875.0,850.0,1.0,35.0,48.18359,11.327474,-4500.0,0.0,0.0,38.56999969482422,0.1940000057220459,194.0,,,not working,Fendt 820,,,Driving Off-Road
0 days 00:00:00.100000,1,2.799999952316284,25.0,575.1,4.699999809265137,47.0,875.0,860.0,1.0,35.0,48.18359,11.327474,-4500.0,0.0,0.0,38.599998474121094,0.22200000286102295,222.0,,,not working,Fen

,Time_(s),index_[-],AccelPedalPos1_(%),ActualEngPercentTorque_(%),Altitude_(m),EngFuelRate_(L/h),EngPercentLoadAtCurrentSpeed_(%),EngRequestedSpeed_SpeedLimit_(RPM),EngSpeed_(RPM),FrontHitchInworkIndication_[-],FrontHitchPos_(%),Latitude_(°),Longitude_(°),RearDraft_(N),RearHitchInWorkIndication_[-],RearHitchPosition_[-],WheelBasedMachineDistance_(m),WheelBasedMachineSpeed_(m/s),WheelBasedSpeed_(RPM),RearPTOOutputShaftSpeed_(RPM),Cluster_[-],WorkType_[-],Tractor_Model_[-],Implement_Model_[-],Implement_Width_(m),Status_[-]
0,0 days 00:00:00,0,2.8,26.0,575.1,4.9,49.0,875.0,850.0,1.0,35.0,48.18359,11.327474,-4500.0,0.0,0.0,38.570000,0.194,194.0,NaN,NaN,not working,Fendt 820,NaN,NaN,Driving Off-Road
1,0 days 00:00:00.100000,1,2.8,25.0,575.1,4.7,47.0,875.0,860.0,1.0,35.0,48.18359,11.327474,-4500.0,0.0,0.0,38.599998,0.222,222.0,NaN,NaN,not working,Fendt 820,NaN,NaN,Driving Off-Road



[5/5] Fendt 211.csv  (519.1 MB)

--- First 3 Lines (Raw Text) ---
Time_(s),EngSpeed_(RPM),ActualEngPercentTorque_(%),DriversDemandEngPercentTorque_(%),EngFuelRate_(L/h),EngReferenceTorque_[Nm],Altitude_(m),Longitude_(°),Latitude_(°),EngPercentLoadAtCurrentSpeed_(%),AccelPedalPos1_(%),EngCoolantPress_(kPa),EngOilPress_(kPa),EngFuelDeliveryPress_(kPa),EngCoolantTemp_(°C),EngIntakeManifold1Temp_(°C),EngIntakeManifold1Press_(kPa),FrontAxleSpeed_(km/h),SpeedOverGround_(m/s),CourseOverGround_(°),AmbientAirTemp_(°C),DEFDoser1AbsPress_(kPa),DEFActualDosingQuantity_(g/h),EngRequestedSpeed_SpeedLimit_(RPM),GroundBasedImplementDistance_[mm],GroundBasedImplementSpeed_[mm/s],Volume_(%),WheelBasedSpeed_(RPM),WheelBasedVehicleSpeed _(km/h),EngChargeAirCooler1OutletTempEngFuelTemp1,NominalFrictionPercentTorque,RearPTOOutputShaftSpeed_(RPM),EstimatedCurvature_(1/km),RearHitchPosition_[-],RearHitchInWorkIndication_[-],RearNominalLowerLinkForce_(%),RearDraft_(N),GroundBasedMachineSpeed_(m/s),GroundBased

,Time_(s),EngSpeed_(RPM),ActualEngPercentTorque_(%),DriversDemandEngPercentTorque_(%),EngFuelRate_(L/h),EngReferenceTorque_[Nm],Altitude_(m),Longitude_(°),Latitude_(°),EngPercentLoadAtCurrentSpeed_(%),AccelPedalPos1_(%),EngCoolantPress_(kPa),EngOilPress_(kPa),EngFuelDeliveryPress_(kPa),EngCoolantTemp_(°C),EngIntakeManifold1Temp_(°C),EngIntakeManifold1Press_(kPa),FrontAxleSpeed_(km/h),SpeedOverGround_(m/s),CourseOverGround_(°),AmbientAirTemp_(°C),DEFDoser1AbsPress_(kPa),DEFActualDosingQuantity_(g/h),EngRequestedSpeed_SpeedLimit_(RPM),GroundBasedImplementDistance_[mm],GroundBasedImplementSpeed_[mm/s],Volume_(%),WheelBasedSpeed_(RPM),WheelBasedVehicleSpeed _(km/h),EngChargeAirCooler1OutletTempEngFuelTemp1,NominalFrictionPercentTorque,RearPTOOutputShaftSpeed_(RPM),EstimatedCurvature_(1/km),RearHitchPosition_[-],RearHitchInWorkIndication_[-],RearNominalLowerLinkForce_(%),RearDraft_(N),GroundBasedMachineSpeed_(m/s),GroundBasedMachineDistance_(m),MachineSelectedSpeed_(m/s),WheelBasedMachineSpeed_(m/s),FrontNominalLowerLinkForce_(%),FrontDraft_(N),LeftStopLights_[-],FrontHitchPos_(%),FrontHitchInworkIndication_[-],RightStopLight,index_[-],Cluster_[-],WorkType_[-],Tractor_Model_[-],Implement_Model_[-],Implement_Width_(m),Status_[-]
0,0 days 00:00:00,801.0,6.0,0.0,1.3,508.0,567.6766,11.327399,48.183502,11.0,2.8,0.0,284.0,392.0,80.25,30.0,2.0,0.0,0.0,260.99945,26.9375,992.0,0.0,800.0,65535.0,65535,37.2,0.0,0.0,NaN,NaN,NaN,-10.984059,98.8,0.0,20.0,12391.161,65.535,65.535,0.0,0.0,104.0,335350.0,0.0,NaN,NaN,NaN,0,NaN,not working,Fendt 211,NaN,NaN,Driving Off-Road
1,0 days 00:00:00.100000,801.5,5.0,0.0,1.3,508.0,567.6766,11.327399,48.183502,11.0,2.8,0.0,284.0,392.0,80.25,30.0,2.0,0.0,0.0,260.99945,26.9375,992.0,0.0,800.0,65535.0,65535,37.2,0.0,0.0,NaN,NaN,NaN,-11.000000,98.8,0.0,20.0,12233.606,65.535,65.535,0.0,0.0,104.0,335350.0,0.0,NaN,NaN,NaN,1,NaN,not working,Fendt 211,NaN,NaN,Driving Off-Road


In [5]:
# --- CELL 4: Canonical Column Name Mapping ---

CANON = {
    # Time & Engine
    "Time": "timestamp",
    "Time_(s)": "timestamp",
    "EngSpeed": "engine_rpm",
    "EngSpeed_(RPM)": "engine_rpm",
    "EngFuelRate": "fuel_rate",
    "EngFuelRate_(L/h)": "fuel_rate",
    
    # Speed
    "WheelBasedMachineSpeed": "veh_speed",
    "WheelBasedMachineSpeed_(m/s)": "veh_speed",
    "SpeedOverGround_(m/s)": "veh_speed",

    # Location
    "Latitude": "latitude",
    "Latitude_(°)": "latitude",
    "Longitude": "longitude",
    "Longitude_(°)": "longitude",
    
    # Load & Hitch
    "ActualEngPercentTorque": "engine_torque",
    "ActualEngPercentTorque_(%)": "engine_torque",
    "RearPTOOutputShaftSpeed": "pto_rpm",
    "RearPTOOutputShaftSpeed_(RPM)": "pto_rpm",
    "RearHitchPosition": "hitch_pos",
    "RearHitchPosition_[-]": "hitch_pos",
    "FrontHitchPos": "front_hitch_pos",
    "FrontHitchPos_(%)": "front_hitch_pos",
    
    # Metadata
    "Cluster_[-]": "cluster_id",
    "Cluster": "cluster_id",
    "WorkType_[-]": "activity_raw",
    "WorkType": "activity_raw",
    "Status_[-]": "status",
    "Status": "status",
}

print("CANON dictionary loaded.")

CANON dictionary loaded.


In [6]:
# --- CELL 5: Clean, Standardize & Convert CSV to Parquet ---

import pandas as pd
import numpy as np
import time
from pathlib import Path

def clean_and_standardize(df):
    """Clean, standardize columns via CANON, and engineer field/activity features."""
    
    # 1. Rename columns
    df = df.rename(columns=CANON)
    
    # 1b. Handle duplicate 'timestamp' columns (from multiple raw names mapping to same canonical name)
    if not df.columns.is_unique:
        if 'timestamp' in df.columns and isinstance(df['timestamp'], pd.DataFrame):
            ts_combined = df['timestamp'].bfill(axis=1).iloc[:, 0]
            df = df.drop(columns=['timestamp'])
            df['timestamp'] = ts_combined
        df = df.loc[:, ~df.columns.duplicated()]

    # 2. Keep only canonical columns
    wanted_cols = list(set(CANON.values()))
    for meta in ['tractor', 'source_file']:
        if meta in df.columns:
            wanted_cols.append(meta)
    existing_cols = [c for c in wanted_cols if c in df.columns]
    df = df[existing_cols].copy()
    
    # --- Feature Engineering ---
    
    # A) Create 'field' from 'cluster_id'
    if 'cluster_id' in df.columns:
        df['cluster_id'] = pd.to_numeric(df['cluster_id'], errors='coerce').fillna(-1).astype(int)
        df['field'] = 'Field_' + df['cluster_id'].astype(str)
    else:
        df['field'] = 'Unknown'

    # B) Standardize 'activity' from raw work type
    if 'activity_raw' in df.columns:
        df['activity'] = df['activity_raw'].fillna('General').astype(str)
        df.drop(columns=['activity_raw'], inplace=True)
    else:
        df['activity'] = 'General'

    # 3. Parse timestamp
    if 'timestamp' in df.columns:
        if df['timestamp'].dtype == object:
            try:
                df['timestamp'] = pd.to_timedelta(df['timestamp']).dt.total_seconds()
            except Exception:
                df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
    
    # 4. Ensure numeric dtypes
    numeric_cols = ['engine_rpm', 'fuel_rate', 'veh_speed', 'latitude', 'longitude', 
                    'pto_rpm', 'engine_torque', 'hitch_pos']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    return df


def process_large_csv_to_parquet(csv_path, output_path, chunk_size=500_000):
    """Read a large CSV in chunks, clean each, and write a single sorted Parquet."""
    print(f"Processing: {csv_path.name} -> {output_path.name}")
    
    temp_files = []
    chunk_count = 0
    total_rows = 0
    
    try:
        with pd.read_csv(csv_path, chunksize=chunk_size, low_memory=False) as reader:
            for i, chunk in enumerate(reader):
                df_clean = clean_and_standardize(chunk)
                
                if 'tractor' not in df_clean.columns:
                    clean_name = csv_path.stem.replace('_processed', '').split('.')[0]
                    df_clean['tractor'] = clean_name
                
                df_clean['source_file'] = csv_path.name
                df_clean.dropna(subset=['timestamp'], inplace=True)
                
                if not df_clean.empty:
                    temp_file = output_path.parent / f"temp_{output_path.stem}_{i}.parquet"
                    df_clean.to_parquet(temp_file, index=False)
                    temp_files.append(temp_file)
                    total_rows += len(df_clean)
                    chunk_count += 1
                    
                    if i % 10 == 0 and i > 0:
                        print(f"   ...chunk {i} ({total_rows:,} rows so far)")

        print(f"Consolidating {chunk_count} chunks...")
        
        if temp_files:
            full_df = pd.concat([pd.read_parquet(f) for f in temp_files], ignore_index=True)
            
            if 'timestamp' in full_df.columns:
                full_df.sort_values(by='timestamp', inplace=True)
            
            # Reorder columns for readability
            priority = ['timestamp', 'tractor', 'activity', 'field', 'fuel_rate', 'veh_speed', 'engine_rpm']
            ordered = [c for c in priority if c in full_df.columns]
            remaining = [c for c in full_df.columns if c not in priority]
            full_df = full_df[ordered + remaining]

            full_df.to_parquet(output_path, index=False)
            for f in temp_files:
                f.unlink()
                
            print(f"Done! {output_path.name}: {total_rows:,} rows, columns: {list(full_df.columns[:5])}...")
            return True
        else:
            return False

    except Exception as e:
        print(f"ERROR processing {csv_path.name}: {e}")
        for f in temp_files:
            if f.exists():
                f.unlink()
        return False

In [7]:
# --- CELL 6: Run Pipeline for All Target Files ---

target_files_names = [
    "Fendt 722.csv",
    "Fendt 314.csv",
    "Fendt 724.csv",
    "Fendt 820.csv",
    "Fendt 211.csv",
]

print(f"Starting pipeline for {len(target_files_names)} target files...")
start_global = time.time()

# Resolve full paths
files_to_process = []
for fname in target_files_names:
    found = list(RAW_DIR.rglob(fname))
    if found:
        files_to_process.append(found[0])
    else:
        print(f"WARNING: {fname} not found in RAW directory.")

# Process sequentially (memory-safe for multi-GB files)
for csv_path in files_to_process:
    output_name = csv_path.stem + "_processed.parquet"
    output_path = PROC_DIR / output_name
    
    success = process_large_csv_to_parquet(csv_path, output_path)
    print(f"-> {'Done' if success else 'FAILED'}: {output_name}\n")

print(f"Pipeline finished in {time.time() - start_global:.1f}s")

Starting pipeline for 5 target files...
Processing: Fendt 722.csv -> Fendt 722_processed.parquet
   ...chunk 10 (5,500,000 rows so far)
   ...chunk 20 (10,500,000 rows so far)
Consolidating 28 chunks...
Done! Fendt 722_processed.parquet: 13,811,758 rows, columns: ['timestamp', 'tractor', 'activity', 'field', 'fuel_rate']...
-> Done: Fendt 722_processed.parquet

Processing: Fendt 314.csv -> Fendt 314_processed.parquet
   ...chunk 10 (5,500,000 rows so far)
Consolidating 17 chunks...
Done! Fendt 314_processed.parquet: 8,143,401 rows, columns: ['timestamp', 'tractor', 'activity', 'field', 'fuel_rate']...
-> Done: Fendt 314_processed.parquet

Processing: Fendt 724.csv -> Fendt 724_processed.parquet
Consolidating 8 chunks...
Done! Fendt 724_processed.parquet: 3,529,865 rows, columns: ['timestamp', 'tractor', 'activity', 'field', 'fuel_rate']...
-> Done: Fendt 724_processed.parquet

Processing: Fendt 820.csv -> Fendt 820_processed.parquet
Consolidating 8 chunks...
Done! Fendt 820_processed.p

In [8]:
# --- CELL 7: Post-Processing Validation ---

processed_files = list(PROC_DIR.glob("*_processed.parquet"))

if not processed_files:
    print("No processed files found for validation.")
else:
    print(f"Found {len(processed_files)} processed files. Validating...")
    
    validation_stats = []
    
    for p_file in processed_files:
        try:
            df = pd.read_parquet(p_file)
            
            ts_is_numeric = pd.api.types.is_numeric_dtype(df['timestamp'])
            nulos = df['timestamp'].isnull().sum()
            min_ts = df['timestamp'].min()
            max_ts = df['timestamp'].max()
            duration = max_ts - min_ts if pd.notnull(min_ts) and pd.notnull(max_ts) else 0
            
            validation_stats.append({
                'File': p_file.name,
                'Rows': len(df),
                'Timestamp Numeric': ts_is_numeric,
                'Nulls': nulos,
                'Duration (s)': duration,
                'Season Span (days)': round(duration / 86400, 1),
            })
            
            if p_file == processed_files[0]:
                print(f"\n--- Sample: {p_file.name} ---")
                display(df.head())
                print("-" * 80)

        except Exception as e:
            print(f"Error reading {p_file.name}: {e}")

    if validation_stats:
        df_stats = pd.DataFrame(validation_stats)
        print("\n--- Validation Report ---")
        if df_stats['Timestamp Numeric'].all():
            print("✅ OK: 'timestamp' is numeric in ALL files.")
        else:
            print("❌ WARNING: Some files have non-numeric timestamps!")
        display(df_stats)

Found 5 processed files. Validating...

--- Sample: Fendt 211_processed.parquet ---


,timestamp,tractor,activity,field,fuel_rate,veh_speed,engine_rpm,latitude,longitude,front_hitch_pos,hitch_pos,pto_rpm,engine_torque,status,cluster_id,source_file
0,0.0,Fendt 211,not working,Field_-1,1.3,0.0,801.00000,48.183502,11.327399,NaN,98.8,NaN,6.0,Driving Off-Road,-1,Fendt 211.csv
1,0.1,Fendt 211,not working,Field_-1,1.3,0.0,801.50000,48.183502,11.327399,NaN,98.8,NaN,5.0,Driving Off-Road,-1,Fendt 211.csv
2,0.2,Fendt 211,not working,Field_-1,1.3,0.0,801.50000,48.183502,11.327399,NaN,98.8,NaN,6.0,Driving Off-Road,-1,Fendt 211.csv
3,0.3,Fendt 211,not working,Field_-1,1.3,0.0,801.50000,48.183502,11.327399,NaN,98.8,NaN,5.0,Driving Off-Road,-1,Fendt 211.csv
4,0.4,Fendt 211,not working,Field_-1,1.3,0.0,801.41846,48.183502,11.327399,NaN,98.8,NaN,5.0,Driving Off-Road,-1,Fendt 211.csv


--------------------------------------------------------------------------------

--- Validation Report ---
✅ OK: 'timestamp' is numeric in ALL files.


,File,Rows,Timestamp Numeric,Nulls,Duration (s),Season Span (days)
0,Fendt 211_processed.parquet,1592890,True,0,2666752.1,30.9
1,Fendt 314_processed.parquet,8143401,True,0,19219054.8,222.4
2,Fendt 722_processed.parquet,13811758,True,0,22465217.1,260.0
3,Fendt 724_processed.parquet,3529865,True,0,6663071.7,77.1
4,Fendt 820_processed.parquet,3955756,True,0,12205347.7,141.3


In [9]:
# --- CELL 8: Consolidate All Processed Files ---

CURATED_DIR = PROJECT_ROOT / "data" / "curated"
CURATED_DIR.mkdir(exist_ok=True, parents=True)

processed_files = list(PROC_DIR.glob("*_processed.parquet"))

if not processed_files:
    print("ERROR: No processed files found for consolidation.")
else:
    print(f"Consolidating {len(processed_files)} files...")
    start_time = time.time()

    df_list = []
    for p_file in processed_files:
        print(f"  Loading: {p_file.name}")
        df_list.append(pd.read_parquet(p_file))
    
    full_df = pd.concat(df_list, ignore_index=True)
    
    # Sort by tractor + time (critical for delta_t computation downstream)
    full_df.sort_values(by=['tractor', 'timestamp'], inplace=True)
    full_df.reset_index(drop=True, inplace=True)

    print(f"Done in {time.time() - start_time:.1f}s")
    print(f"Total rows: {len(full_df):,}")
    print(f"Tractors: {full_df['tractor'].unique()}")

Consolidating 5 files...
  Loading: Fendt 211_processed.parquet
  Loading: Fendt 314_processed.parquet
  Loading: Fendt 722_processed.parquet
  Loading: Fendt 724_processed.parquet
  Loading: Fendt 820_processed.parquet
Done in 18.7s
Total rows: 31,033,670
Tractors: ['Fendt 211' 'Fendt 314' 'Fendt 722' 'Fendt 724' 'Fendt 820']


In [10]:
# --- CELL 8.1: Exploratory Data Analysis (EDA) ---

if 'full_df' not in locals():
    print("Warning: Run previous cells to load full_df.")

import plotly.express as px

print("\n--- EXPLORATORY DATA ANALYSIS (EDA) ---\n")

# Sampling rate: 10 Hz (0.1s intervals)
POINTS_PER_SECOND = 10
POINTS_PER_HOUR = 3600 * POINTS_PER_SECOND

print(f"Sampling rate: {POINTS_PER_SECOND} Hz")
print(f"Points-to-hours divisor: {POINTS_PER_HOUR}\n")

# === 1. Fleet Summary (operating hours) ===
print("## 1. Fleet Summary (Operating Hours)\n")

df_fleet = full_df['tractor'].value_counts().reset_index()
df_fleet.columns = ['Tractor', 'Total Datapoints']
df_fleet['Total Hours (h)'] = df_fleet['Total Datapoints'] / POINTS_PER_HOUR
df_fleet['Share (%)'] = (df_fleet['Total Hours (h)'] / df_fleet['Total Hours (h)'].sum()) * 100

df_fleet_print = df_fleet.copy()
df_fleet_print['Total Hours (h)'] = df_fleet_print['Total Hours (h)'].map('{:,.1f}'.format)
df_fleet_print['Share (%)'] = df_fleet_print['Share (%)'].round(2)

print(df_fleet_print[['Tractor', 'Total Hours (h)', 'Share (%)']].to_string(index=False))
print(f"\nTotal fleet hours: {df_fleet['Total Hours (h)'].sum():,.1f} h")
print("\n" + "=" * 80 + "\n")

# === 2. Activity Distribution (hours) ===
print("## 2. Activity Distribution (Labeled Hours)\n")

df_activity = full_df['activity'].value_counts().reset_index()
df_activity.columns = ['Activity', 'Total Datapoints']
df_activity['Total Hours (h)'] = df_activity['Total Datapoints'] / POINTS_PER_HOUR
df_activity['Share (%)'] = (df_activity['Total Hours (h)'] / df_activity['Total Hours (h)'].sum()) * 100

df_act_print = df_activity.copy()
df_act_print['Total Hours (h)'] = df_act_print['Total Hours (h)'].map('{:,.1f}'.format)
df_act_print['Share (%)'] = df_act_print['Share (%)'].round(2)

print(df_act_print[['Activity', 'Total Hours (h)', 'Share (%)']].to_string(index=False))

fig_act = px.bar(df_activity,
                 x='Activity', y='Total Hours (h)',
                 title='Total Hours by Labeled Activity (10 Hz corrected)',
                 color='Total Hours (h)',
                 color_continuous_scale=px.colors.sequential.Viridis)
fig_act.show()
print("\n" + "=" * 80 + "\n")

# === 3 & 4. Boxplots (sampled for performance) ===
print("## Preparing sample for boxplots...")
SAMPLE_SIZE = 100_000

if len(full_df) > SAMPLE_SIZE:
    df_sample = full_df.sample(n=SAMPLE_SIZE, random_state=42).copy()
else:
    df_sample = full_df.copy()

df_sample['fuel_rate'] = pd.to_numeric(df_sample['fuel_rate'], errors='coerce')
df_sample['veh_speed'] = pd.to_numeric(df_sample['veh_speed'], errors='coerce')

top_10_activities = df_activity['Activity'].head(10).tolist()
df_viz = df_sample[df_sample['activity'].isin(top_10_activities)]
print("Sample ready.\n")

# 3. Fuel rate by activity
print("## 3. Fuel Rate by Activity\n")
fig_box = px.box(df_viz, x='activity', y='fuel_rate', color='activity',
                 title=f'Fuel Rate Distribution (L/h) — {SAMPLE_SIZE:,} sample',
                 points="outliers")
fig_box.update_yaxes(range=[0, df_viz['fuel_rate'].quantile(0.995)])
fig_box.show()
print("\n" + "=" * 80 + "\n")

# 4. Speed by activity
print("## 4. Vehicle Speed by Activity\n")
fig_speed = px.box(df_viz, x='activity', y='veh_speed', color='activity',
                   title=f'Speed Distribution (m/s) — {SAMPLE_SIZE:,} sample',
                   points="outliers")
fig_speed.update_yaxes(range=[0, df_viz['veh_speed'].quantile(0.995)])
fig_speed.show()


--- EXPLORATORY DATA ANALYSIS (EDA) ---

Sampling rate: 10 Hz
Points-to-hours divisor: 36000

## 1. Fleet Summary (Operating Hours)

  Tractor Total Hours (h)  Share (%)
Fendt 722           383.7      44.51
Fendt 314           226.2      26.24
Fendt 820           109.9      12.75
Fendt 724            98.1      11.37
Fendt 211            44.2       5.13

Total fleet hours: 862.0 h


## 2. Activity Distribution (Labeled Hours)

              Activity Total Hours (h)  Share (%)
           not working           516.7      59.94
Seed drill combination            99.6      11.55
             Ploughing            55.7       6.46
    Cultivating (deep)            49.8       5.77
             Transport            29.2       3.39
              Mulching            27.7       3.21
   Seedbed combination            23.7       2.75
 Cultivating (shallow)            15.4       1.79
        Disc harrowing            12.3       1.43
           Fertilizing            11.9       1.38
       Power harrow



## Preparing sample for boxplots...
Sample ready.

## 3. Fuel Rate by Activity





## 4. Vehicle Speed by Activity



In [24]:
# --- CELL 9: Fuel Rate by Tractor (ANOVA + Tukey, balanced 5-min blocks) ---

import scipy.stats as stats
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc
from statsmodels.stats.multicomp import pairwise_tukeyhsd


def get_cld_letters(tukey_results, means_sorted):
    """Generate compact letter display (a, b, c...) from Tukey HSD results."""
    labels = means_sorted.index.tolist()
    letters = {l: '' for l in labels}
    df_res = pd.DataFrame(data=tukey_results.summary().data[1:],
                          columns=tukey_results.summary().data[0])
    
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    not_assigned = labels.copy()
    letter_idx = 0
    
    while not_assigned:
        base = not_assigned[0]
        letter = alphabet[letter_idx]
        letters[base] += letter
        
        for other in not_assigned[1:]:
            pair = df_res[((df_res['group1'] == base) & (df_res['group2'] == other)) |
                          ((df_res['group1'] == other) & (df_res['group2'] == base))]
            if not pair.empty and not pair['reject'].values[0]:
                letters[other] += letter
        
        not_assigned.pop(0)
        letter_idx += 1
    
    return letters


BLOCK_SIZE = 28800  # 8 hours (one day) min at 10 Hz
print("--- Fuel Rate Mean Comparison (Tractors) — Balanced 5-min blocks ---\n")

if 'full_df' in locals():
    # 1. Filter engine-on rows
    df_stats = full_df[(full_df['engine_rpm'] > 0) & (full_df['fuel_rate'].notnull())].copy()

    # 2. Aggregate into 5-min blocks per tractor
    blocks = []
    for tractor, grp in df_stats.groupby('tractor'):
        grp_sorted = grp.sort_values('timestamp')
        n_blocks = len(grp_sorted) // BLOCK_SIZE
        for i in range(n_blocks):
            chunk = grp_sorted.iloc[i * BLOCK_SIZE : (i + 1) * BLOCK_SIZE]
            blocks.append({
                'tractor': tractor,
                'fuel_rate': chunk['fuel_rate'].mean(),
            })

    df_blocks = pd.DataFrame(blocks)

    # 3. Balance: subsample to the smallest group
    group_counts = df_blocks.groupby('tractor').size()
    min_n = group_counts.min()
    limiting = group_counts.idxmin()
    print(f"Block replicates before balancing:")
    print(group_counts.to_string())
    print(f"\nBalancing all groups to n = {min_n} (limited by {limiting})\n")

    df_balanced = (
        df_blocks
        .groupby('tractor', group_keys=False)
        .apply(lambda g: g.sample(n=min_n, random_state=42))
        .reset_index(drop=True)
    )

    # 4. ANOVA
    groups = [d['fuel_rate'].values for _, d in df_balanced.groupby('tractor')]
    f_stat, p_value = stats.f_oneway(*groups)
    print(f"ANOVA F = {f_stat:.2f} | p = {p_value:.4e}")
    print(f"Total replicates: {len(df_balanced)} ({min_n} per tractor × {df_balanced['tractor'].nunique()} tractors)")
    print(f"Significance level: α = 0.001")

    if p_value < 0.01:
        # 5. Tukey HSD + CLD
        tukey = pairwise_tukeyhsd(endog=df_balanced['fuel_rate'], groups=df_balanced['tractor'], alpha=0.001)
        summary = df_balanced.groupby('tractor')['fuel_rate'].agg(['mean', 'std']).sort_values('mean', ascending=False)
        letter_dict = get_cld_letters(tukey, summary['mean'])
        summary['Group'] = summary.index.map(letter_dict)

        print(f"\n=== Mean Fuel Rate by Tractor (n = {min_n} blocks each) ===")
        print(f"{'Tractor':<15} | {'Mean ± SD (L/h)':<20} | {'Group'}")
        print("-" * 55)
        for tractor, row in summary.iterrows():
            print(f"{tractor:<15} | {row['mean']:.2f} ± {row['std']:.2f}         | {row['Group']}")
        print("-" * 55)

        # 6. Boxplot (greyscale)
        n_colors = len(summary)
        custom_greys = pc.sample_colorscale('Greys', np.linspace(1.0, 0.25, n_colors))

        fig = px.box(
            df_balanced, x='tractor', y='fuel_rate', color='tractor',
            category_orders={'tractor': summary.index.tolist()},
            color_discrete_sequence=custom_greys,
            title=f"Fuel Rate by Tractor (balanced: n={min_n} blocks of 5 min each)",
            labels={'fuel_rate': 'Fuel Rate (L/h)', 'tractor': 'Model'},
            points=False,
        )
        fig.add_trace(go.Scatter(
            x=summary.index, y=summary['mean'],
            mode='markers', marker=dict(color='red', size=8, symbol='diamond'),
            name='Mean',
        ))
        fig.update_layout(showlegend=False, height=500, template='plotly_white',
                          font=dict(color="black"))
        fig.show()
    else:
        print("No significant differences found.")
else:
    print("ERROR: full_df not loaded.")

--- Fuel Rate Mean Comparison (Tractors) — Balanced 5-min blocks ---

Block replicates before balancing:
tractor
Fendt 211     55
Fendt 314    282
Fendt 722    386
Fendt 724    122
Fendt 820    137

Balancing all groups to n = 55 (limited by Fendt 211)

ANOVA F = 37.11 | p = 1.0024e-24
Total replicates: 275 (55 per tractor × 5 tractors)
Significance level: α = 0.001

=== Mean Fuel Rate by Tractor (n = 55 blocks each) ===
Tractor         | Mean ± SD (L/h)      | Group
-------------------------------------------------------
Fendt 820       | 17.62 ± 7.22         | a
Fendt 722       | 16.48 ± 8.28         | ab
Fendt 724       | 12.92 ± 7.76         | abc
Fendt 314       | 9.70 ± 4.63         | cd
Fendt 211       | 4.68 ± 1.78         | e
-------------------------------------------------------


In [25]:
# --- CELL 9b: Fuel Rate by Activity (ANOVA + Tukey, balanced 5-min blocks) ---

BLOCK_SIZE = 28800  # 5 min at 10 Hz
MIN_HOURS = 1.0    # Drop activities with less than 1 hour total
print("--- Fuel Rate Mean Comparison (Activities) — Balanced 8-hour blocks ---\n")

if 'full_df' in locals():
    # 1. Filter engine-on rows
    df_stats = full_df[(full_df['engine_rpm'] > 0) & (full_df['fuel_rate'].notnull())].copy()

    # 2. Aggregate into 5-min blocks per activity
    blocks = []
    for activity, grp in df_stats.groupby('activity'):
        grp_sorted = grp.sort_values('timestamp')
        n_blocks = len(grp_sorted) // BLOCK_SIZE
        if n_blocks < 1:
            continue
        for i in range(n_blocks):
            chunk = grp_sorted.iloc[i * BLOCK_SIZE : (i + 1) * BLOCK_SIZE]
            blocks.append({
                'activity': activity,
                'fuel_rate': chunk['fuel_rate'].mean(),
            })

    df_blocks = pd.DataFrame(blocks)

    # 3. Drop activities with too few blocks (< MIN_HOURS worth)
    min_blocks = int(MIN_HOURS * 12)  # 12 blocks per hour (5 min each)
    group_counts = df_blocks.groupby('activity').size()
    valid = group_counts[group_counts >= min_blocks].index
    df_blocks = df_blocks[df_blocks['activity'].isin(valid)]

    # 4. Balance: subsample to smallest group
    group_counts = df_blocks.groupby('activity').size()
    min_n = group_counts.min()
    limiting = group_counts.idxmin()
    print(f"Block replicates before balancing (activities ≥ {min_blocks} blocks):")
    print(group_counts.sort_values(ascending=False).to_string())
    print(f"\nBalancing all groups to n = {min_n} (limited by {limiting})\n")

    df_balanced = (
        df_blocks
        .groupby('activity', group_keys=False)
        .apply(lambda g: g.sample(n=min_n, random_state=42))
        .reset_index(drop=True)
    )

    # 5. ANOVA
    groups = [d['fuel_rate'].values for _, d in df_balanced.groupby('activity')]
    f_stat, p_value = stats.f_oneway(*groups)
    print(f"ANOVA F = {f_stat:.2f} | p = {p_value:.4e}")
    print(f"Significance level: α = 0.001")
    print(f"\n=== Mean Fuel Rate by Activity (n = {min_n} blocks each, Tukey HSD α = 0.001) ===")
        
    if p_value < 0.01:
        # 6. Tukey HSD + CLD
        tukey = pairwise_tukeyhsd(endog=df_balanced['fuel_rate'], groups=df_balanced['activity'], alpha=0.001)
        summary = df_balanced.groupby('activity')['fuel_rate'].agg(['mean', 'std']).sort_values('mean', ascending=False)
        letter_dict = get_cld_letters(tukey, summary['mean'])
        summary['Group'] = summary.index.map(letter_dict)

        print(f"\n=== Mean Fuel Rate by Activity (n = {min_n} blocks each) ===")
        print(f"{'Activity':<30} | {'Mean ± SD (L/h)':<20} | {'Group'}")
        print("-" * 70)
        for activity, row in summary.iterrows():
            print(f"{activity:<30} | {row['mean']:.2f} ± {row['std']:.2f}         | {row['Group']}")
        print("-" * 70)

        # 7. Boxplot (greyscale)
        n_colors = len(summary)
        custom_greys = pc.sample_colorscale('Greys', np.linspace(1.0, 0.25, n_colors))

        fig = px.box(
            df_balanced, x='activity', y='fuel_rate', color='activity',
            category_orders={'activity': summary.index.tolist()},
            color_discrete_sequence=custom_greys,
            title=f"Fuel Rate by Activity (balanced: n={min_n} blocks of 5 min each)",
            labels={'fuel_rate': 'Fuel Rate (L/h)', 'activity': 'Activity'},
            points=False,
        )
        fig.add_trace(go.Scatter(
            x=summary.index, y=summary['mean'],
            mode='markers', marker=dict(color='red', size=8, symbol='diamond'),
            name='Mean',
        ))
        fig.update_layout(showlegend=False, height=500, template='plotly_white',
                          font=dict(color="black"),
                          xaxis_tickangle=-35)
        fig.show()
    else:
        print("No significant differences found.")
else:
    print("ERROR: full_df not loaded.")

--- Fuel Rate Mean Comparison (Activities) — Balanced 8-hour blocks ---

Block replicates before balancing (activities ≥ 12 blocks):
activity
not working               553
Seed drill combination    124
Ploughing                  69
Cultivating (deep)         62
Transport                  36
Mulching                   34
Seedbed combination        29
Cultivating (shallow)      19
Disc harrowing             15
Fertilizing                14

Balancing all groups to n = 14 (limited by Fertilizing)

ANOVA F = 19.26 | p = 3.7605e-20
Significance level: α = 0.001

=== Mean Fuel Rate by Activity (n = 14 blocks each, Tukey HSD α = 0.001) ===

=== Mean Fuel Rate by Activity (n = 14 blocks each) ===
Activity                       | Mean ± SD (L/h)      | Group
----------------------------------------------------------------------
Cultivating (deep)             | 27.36 ± 6.82         | a
Ploughing                      | 25.91 ± 7.82         | ab
Disc harrowing                 | 24.30 ± 3.40       

In [13]:
# --- CELL 10: Idle Analysis by Tractor (Stationary + Low RPM) ---

THRESHOLD_SPEED_MS = 0.75   # m/s (~2.7 km/h)
IDLE_RPM_MIN = 700
IDLE_RPM_MAX = 1000
FREQ_HZ = 10
POINTS_PER_HOUR = 3600 * FREQ_HZ

print(f"--- Idle Analysis (speed < {THRESHOLD_SPEED_MS} m/s & {IDLE_RPM_MIN} < RPM < {IDLE_RPM_MAX}) ---\n")

if 'full_df' in locals():
    df_engine_on = full_df[full_df['engine_rpm'] > IDLE_RPM_MIN].copy()

    df_engine_on['is_idle'] = (
        (df_engine_on['veh_speed'] < THRESHOLD_SPEED_MS) &
        (df_engine_on['engine_rpm'] < IDLE_RPM_MAX)
    )

    stats_tractor = df_engine_on.groupby('tractor')['is_idle'].agg(['count', 'sum']).reset_index()
    stats_tractor.columns = ['Tractor', 'Total Points', 'Idle Points']

    stats_tractor['Engine-On Hours (h)'] = stats_tractor['Total Points'] / POINTS_PER_HOUR
    stats_tractor['Idle Hours (h)'] = stats_tractor['Idle Points'] / POINTS_PER_HOUR
    stats_tractor['Idle (%)'] = (stats_tractor['Idle Hours (h)'] / stats_tractor['Engine-On Hours (h)']) * 100

    stats_tractor = stats_tractor.sort_values('Idle (%)', ascending=False)

    print("=== IDLE RANKING BY TRACTOR ===")
    print(f"{'Tractor':<15} | {'Engine-On (h)':<15} | {'Idle (h)':<15} | {'Idle (%)'}")
    print("-" * 65)
    for _, row in stats_tractor.iterrows():
        print(f"{row['Tractor']:<15} | {row['Engine-On Hours (h)']:<15.1f} | {row['Idle Hours (h)']:<15.1f} | {row['Idle (%)']:.1f}%")
    print("-" * 65)
    print(f"Total engine-on hours analyzed: {stats_tractor['Engine-On Hours (h)'].sum():,.1f} h")
else:
    print("ERROR: full_df not loaded.")

--- Idle Analysis (speed < 0.75 m/s & 700 < RPM < 1000) ---

=== IDLE RANKING BY TRACTOR ===
Tractor         | Engine-On (h)   | Idle (h)        | Idle (%)
-----------------------------------------------------------------
Fendt 724       | 98.0            | 42.9            | 43.8%
Fendt 314       | 225.9           | 68.4            | 30.3%
Fendt 211       | 44.2            | 12.0            | 27.2%
Fendt 820       | 109.8           | 15.9            | 14.5%
Fendt 722       | 295.3           | 38.7            | 13.1%
-----------------------------------------------------------------
Total engine-on hours analyzed: 773.3 h


In [14]:
# --- CELL 11: Idle Breakdown by Activity ---

print("--- Idle Time Breakdown by Activity ---\n")

if 'full_df' in locals() and 'df_engine_on' in locals():
    stats_activity = df_engine_on.groupby('activity')['is_idle'].agg(['count', 'sum']).reset_index()
    stats_activity.columns = ['Activity', 'Total Points', 'Idle Points']

    stats_activity['Total Hours (h)'] = stats_activity['Total Points'] / POINTS_PER_HOUR
    stats_activity['Idle Hours (h)'] = stats_activity['Idle Points'] / POINTS_PER_HOUR
    stats_activity['Idle (%)'] = (stats_activity['Idle Hours (h)'] / stats_activity['Total Hours (h)']) * 100

    # Filter out activities with < 10h to reduce noise
    stats_activity = stats_activity[stats_activity['Total Hours (h)'] > 10].sort_values('Idle (%)', ascending=False)

    print("=== IDLE RATE BY ACTIVITY (>10 h only) ===")
    print(f"{'Activity':<30} | {'Total (h)':<12} | {'Idle (h)':<12} | {'Idle (%)'}")
    print("-" * 75)
    for _, row in stats_activity.iterrows():
        flag = "(!)" if row['Idle (%)'] > 30 else ""
        print(f"{row['Activity']:<30} | {row['Total Hours (h)']:<12.1f} | {row['Idle Hours (h)']:<12.1f} | {row['Idle (%)']:.1f}% {flag}")
    print("-" * 75)
    print(f"* Idle = speed < {THRESHOLD_SPEED_MS} m/s & {IDLE_RPM_MIN} < RPM < {IDLE_RPM_MAX}")
    print("* Activities under 10 h total omitted.")
else:
    print("ERROR: Run previous cell first to generate 'df_engine_on'.")

--- Idle Time Breakdown by Activity ---

=== IDLE RATE BY ACTIVITY (>10 h only) ===
Activity                       | Total (h)    | Idle (h)     | Idle (%)
---------------------------------------------------------------------------
not working                    | 428.0        | 160.4        | 37.5% (!)
Transport                      | 29.2         | 8.5          | 29.3% 
Seedbed combination            | 23.7         | 1.7          | 7.3% 
Cultivating (deep)             | 49.8         | 1.9          | 3.9% 
Ploughing                      | 55.7         | 1.6          | 2.9% 
Mulching                       | 27.7         | 0.8          | 2.7% 
Cultivating (shallow)          | 15.4         | 0.4          | 2.5% 
Seed drill combination         | 99.6         | 2.1          | 2.1% 
Fertilizing                    | 11.9         | 0.2          | 1.6% 
Disc harrowing                 | 12.3         | 0.1          | 0.9% 
-------------------------------------------------------------------------

In [23]:
# --- CELL 9: Fuel Rate by Tractor (ANOVA + Tukey, balanced blocks) ---

import scipy.stats as stats
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc
from statsmodels.stats.multicomp import pairwise_tukeyhsd


def get_cld_letters(tukey_results, means_sorted):
    """Generate compact letter display (a, b, c...) from Tukey HSD results."""
    labels = means_sorted.index.tolist()
    letters = {l: '' for l in labels}
    df_res = pd.DataFrame(data=tukey_results.summary().data[1:],
                          columns=tukey_results.summary().data[0])
    
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    not_assigned = labels.copy()
    letter_idx = 0
    
    while not_assigned:
        base = not_assigned[0]
        letter = alphabet[letter_idx]
        letters[base] += letter
        
        for other in not_assigned[1:]:
            pair = df_res[((df_res['group1'] == base) & (df_res['group2'] == other)) |
                          ((df_res['group1'] == other) & (df_res['group2'] == base))]
            if not pair.empty and not pair['reject'].values[0]:
                letters[other] += letter
        
        not_assigned.pop(0)
        letter_idx += 1
    
    return letters


BLOCK_SIZE = 28800  # 8 hours at 10 Hz
ALPHA = 0.001
print("--- Comparação de Médias de Consumo por Trator — Blocos balanceados de 8h ---\n")

if 'full_df' in locals():
    # 1. Filtrar motor ligado
    df_stats = full_df[(full_df['engine_rpm'] > 0) & (full_df['fuel_rate'].notnull())].copy()

    # 2. Agregar em blocos de 8h por trator
    blocks = []
    for tractor, grp in df_stats.groupby('tractor'):
        grp_sorted = grp.sort_values('timestamp')
        n_blocks = len(grp_sorted) // BLOCK_SIZE
        for i in range(n_blocks):
            chunk = grp_sorted.iloc[i * BLOCK_SIZE : (i + 1) * BLOCK_SIZE]
            blocks.append({
                'tractor': tractor,
                'fuel_rate': chunk['fuel_rate'].mean(),
            })

    df_blocks = pd.DataFrame(blocks)

    # 3. Balancear: subamostra pelo menor grupo
    group_counts = df_blocks.groupby('tractor').size()
    min_n = group_counts.min()
    limiting = group_counts.idxmin()
    print(f"Repetições por trator (antes do balanceamento):")
    print(group_counts.to_string())
    print(f"\nBalanceando todos os grupos para n = {min_n} (limitado por {limiting})\n")

    df_balanced = (
        df_blocks
        .groupby('tractor', group_keys=False)
        .apply(lambda g: g.sample(n=min_n, random_state=42))
        .reset_index(drop=True)
    )

    # 4. ANOVA
    groups = [d['fuel_rate'].values for _, d in df_balanced.groupby('tractor')]
    f_stat, p_value = stats.f_oneway(*groups)
    print(f"ANOVA F = {f_stat:.2f} | p = {p_value:.4e}")
    print(f"Nível de significância: α = {ALPHA}")
    print(f"Total de repetições: {len(df_balanced)} ({min_n} por trator × {df_balanced['tractor'].nunique()} tratores)")

    if p_value < ALPHA:
        # 5. Tukey HSD + CLD
        tukey = pairwise_tukeyhsd(endog=df_balanced['fuel_rate'], groups=df_balanced['tractor'], alpha=ALPHA)
        summary = df_balanced.groupby('tractor')['fuel_rate'].agg(['mean', 'std']).sort_values('mean', ascending=False)
        letter_dict = get_cld_letters(tukey, summary['mean'])
        summary['Group'] = summary.index.map(letter_dict)

        print(f"\n=== Consumo Médio por Trator (n = {min_n} blocos, Tukey HSD α = {ALPHA}) ===")
        print(f"{'Trator':<15} | {'Média ± DP (L/h)':<20} | {'Grupo'}")
        print("-" * 55)
        for tractor, row in summary.iterrows():
            print(f"{tractor:<15} | {row['mean']:.2f} ± {row['std']:.2f}         | {row['Group']}")
        print("-" * 55)

        # 6. Boxplot (escala de cinza)
        n_colors = len(summary)
        custom_greys = pc.sample_colorscale('Greys', np.linspace(1.0, 0.25, n_colors))

        fig = px.box(
            df_balanced, x='tractor', y='fuel_rate', color='tractor',
            category_orders={'tractor': summary.index.tolist()},
            color_discrete_sequence=custom_greys,
            title=f"Distribuição do Consumo por Trator (n={min_n} blocos de 8h, ANOVA p < 0,0001)",
            labels={'fuel_rate': 'Consumo (L/h)', 'tractor': 'Modelo'},
            points=False,
        )
        fig.add_trace(go.Scatter(
            x=summary.index, y=summary['mean'],
            mode='markers', marker=dict(color='red', size=8, symbol='diamond'),
            name='Média',
        ))
        fig.update_layout(showlegend=False, height=500, template='plotly_white',
                          font=dict(color="black"))
        fig.show()
    else:
        print("Nenhuma diferença significativa encontrada.")
else:
    print("ERRO: full_df não carregado.")

--- Comparação de Médias de Consumo por Trator — Blocos balanceados de 8h ---

Repetições por trator (antes do balanceamento):
tractor
Fendt 211     55
Fendt 314    282
Fendt 722    386
Fendt 724    122
Fendt 820    137

Balanceando todos os grupos para n = 55 (limitado por Fendt 211)

ANOVA F = 37.11 | p = 1.0024e-24
Nível de significância: α = 0.001
Total de repetições: 275 (55 por trator × 5 tratores)

=== Consumo Médio por Trator (n = 55 blocos, Tukey HSD α = 0.001) ===
Trator          | Média ± DP (L/h)     | Grupo
-------------------------------------------------------
Fendt 820       | 17.62 ± 7.22         | a
Fendt 722       | 16.48 ± 8.28         | ab
Fendt 724       | 12.92 ± 7.76         | abc
Fendt 314       | 9.70 ± 4.63         | cd
Fendt 211       | 4.68 ± 1.78         | e
-------------------------------------------------------


In [22]:
# --- CELL 9b: Fuel Rate by Activity (ANOVA + Tukey, balanced blocks) ---

BLOCK_SIZE = 28800  # 8 hours at 10 Hz
ALPHA = 0.001
MIN_HOURS = 8.0     # Mínimo de 1 bloco completo
print("--- Comparação de Médias de Consumo por Atividade — Blocos balanceados de 8h ---\n")

# Dicionário de tradução das atividades
ACTIVITY_PT = {
    'Ploughing': 'Aração',
    'Cultivating (deep)': 'Cultivo Profundo',
    'Cultivating (shallow)': 'Cultivo Superficial',
    'Disc harrowing': 'Gradeação',
    'Power harrowing': 'Grade Rotativa',
    'Seedbed combination': 'Prep. Canteiro',
    'Seed drill combination': 'Semeadura Conj.',
    'Seed drill combination 3m': 'Semeadura Conj. 3m',
    'Seed drill combination 4m': 'Semeadura Conj. 4m',
    'Fertilizing': 'Adubação',
    'Spraying': 'Pulverização',
    'Mulching': 'Trituração',
    'Mowing (front)': 'Roçada (frontal)',
    'Mowing (large-scale)': 'Roçada (larga)',
    'Swathing': 'Enleiramento',
    'Transport': 'Transporte',
    'Rotary tilling': 'Enxada Rotativa',
    'Precision air seeding': 'Semeadura Pneumática',
    'not working': 'Ocioso/Parado',
}

if 'full_df' in locals():
    # 1. Filtrar motor ligado
    df_stats = full_df[(full_df['engine_rpm'] > 0) & (full_df['fuel_rate'].notnull())].copy()

    # 2. Agregar em blocos de 8h por atividade
    blocks = []
    for activity, grp in df_stats.groupby('activity'):
        grp_sorted = grp.sort_values('timestamp')
        n_blocks = len(grp_sorted) // BLOCK_SIZE
        if n_blocks < 1:
            continue
        for i in range(n_blocks):
            chunk = grp_sorted.iloc[i * BLOCK_SIZE : (i + 1) * BLOCK_SIZE]
            blocks.append({
                'activity': activity,
                'fuel_rate': chunk['fuel_rate'].mean(),
            })

    df_blocks = pd.DataFrame(blocks)

    # 3. Filtrar atividades com blocos insuficientes
    min_blocks = max(1, int(MIN_HOURS * 3600 * 10 / BLOCK_SIZE))
    group_counts = df_blocks.groupby('activity').size()
    valid = group_counts[group_counts >= min_blocks].index
    df_blocks = df_blocks[df_blocks['activity'].isin(valid)]

    # 4. Balancear
    group_counts = df_blocks.groupby('activity').size()
    min_n = group_counts.min()
    limiting = group_counts.idxmin()
    print(f"Repetições por atividade (antes do balanceamento, ≥ {min_blocks} blocos):")
    print(group_counts.sort_values(ascending=False).to_string())
    print(f"\nBalanceando todos os grupos para n = {min_n} (limitado por {ACTIVITY_PT.get(limiting, limiting)})\n")

    df_balanced = (
        df_blocks
        .groupby('activity', group_keys=False)
        .apply(lambda g: g.sample(n=min_n, random_state=42))
        .reset_index(drop=True)
    )

    # 5. Traduzir atividades para português
    df_balanced['atividade'] = df_balanced['activity'].map(ACTIVITY_PT).fillna(df_balanced['activity'])

    # 6. ANOVA
    groups = [d['fuel_rate'].values for _, d in df_balanced.groupby('atividade')]
    f_stat, p_value = stats.f_oneway(*groups)
    print(f"ANOVA F = {f_stat:.2f} | p = {p_value:.4e}")
    print(f"Nível de significância: α = {ALPHA}")
    print(f"Total de repetições: {len(df_balanced)} ({min_n} por atividade × {df_balanced['atividade'].nunique()} atividades)")

    if p_value < ALPHA:
        # 7. Tukey HSD + CLD
        tukey = pairwise_tukeyhsd(endog=df_balanced['fuel_rate'], groups=df_balanced['atividade'], alpha=ALPHA)
        summary = df_balanced.groupby('atividade')['fuel_rate'].agg(['mean', 'std']).sort_values('mean', ascending=False)
        letter_dict = get_cld_letters(tukey, summary['mean'])
        summary['Group'] = summary.index.map(letter_dict)

        print(f"\n=== Consumo Médio por Atividade (n = {min_n} blocos, Tukey HSD α = {ALPHA}) ===")
        print(f"{'Atividade':<30} | {'Média ± DP (L/h)':<20} | {'Grupo'}")
        print("-" * 70)
        for atividade, row in summary.iterrows():
            print(f"{atividade:<30} | {row['mean']:.2f} ± {row['std']:.2f}         | {row['Group']}")
        print("-" * 70)

        # 8. Boxplot (escala de cinza)
        n_colors = len(summary)
        custom_greys = pc.sample_colorscale('Greys', np.linspace(1.0, 0.25, n_colors))

        fig = px.box(
            df_balanced, x='atividade', y='fuel_rate', color='atividade',
            category_orders={'atividade': summary.index.tolist()},
            color_discrete_sequence=custom_greys,
            title=f"Distribuição do Consumo por Atividade (n={min_n} blocos de 8h, ANOVA p < 0,0001)",
            labels={'fuel_rate': 'Consumo (L/h)', 'atividade': 'Atividade Agrícola'},
            points=False,
        )
        fig.add_trace(go.Scatter(
            x=summary.index, y=summary['mean'],
            mode='markers', marker=dict(color='red', size=8, symbol='diamond'),
            name='Média',
        ))
        fig.update_layout(showlegend=False, height=500, template='plotly_white',
                          font=dict(color="black"),
                          xaxis_tickangle=-35)
        fig.show()
    else:
        print("Nenhuma diferença significativa encontrada.")
else:
    print("ERRO: full_df não carregado.")

--- Comparação de Médias de Consumo por Atividade — Blocos balanceados de 8h ---

Repetições por atividade (antes do balanceamento, ≥ 10 blocos):
activity
not working               553
Seed drill combination    124
Ploughing                  69
Cultivating (deep)         62
Transport                  36
Mulching                   34
Seedbed combination        29
Cultivating (shallow)      19
Disc harrowing             15
Fertilizing                14

Balanceando todos os grupos para n = 14 (limitado por Adubação)

ANOVA F = 19.26 | p = 3.7605e-20
Nível de significância: α = 0.001
Total de repetições: 140 (14 por atividade × 10 atividades)

=== Consumo Médio por Atividade (n = 14 blocos, Tukey HSD α = 0.001) ===
Atividade                      | Média ± DP (L/h)     | Grupo
----------------------------------------------------------------------
Cultivo Profundo               | 27.36 ± 6.82         | a
Aração                         | 25.91 ± 7.82         | ab
Gradeação                   